In [2]:
import os
import sys
sys.path.append(os.path.abspath(".."))

In [3]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from app.retrieval.bm25 import BM25Retriever
from app.retrieval.faiss_retriever import FaissRetriever
from app.retrieval.hybrid import HybridRetriever

In [4]:
df = pd.read_csv("../data/processed/catalog_processed.csv")
df.head()

,entity_id,name,link,scraped_at,job_levels,job_levels_raw,languages,languages_raw,duration,duration_raw,...,remote,adaptive,description,keys,search_document,duration_minutes,is_report,assessment_type,job_level,language
0,4302,Global Skills Development Report,https://www.shl.com/products/product-catalog/v...,2026-05-08T10:40:21.464836+00:00,"['Director', 'Entry-Level', 'Executive', 'Gene...","Director, Entry-Level, Executive, General Popu...",[],NaN,NaN,NaN,...,yes,no,This report is designed to be given to individ...,"['Ability & Aptitude', 'Assessment Exercises',...",Name: Global Skills Development Report\nDescri...,NaN,True,"Ability & Aptitude, Assessment Exercises, Biod...","Director, Entry-Level, Executive, General Popu...",NaN
1,3827,.NET Framework 4.5,https://www.shl.com/products/product-catalog/v...,2026-05-08T10:39:46.810448+00:00,"['Professional Individual Contributor', 'Mid-P...","Professional Individual Contributor, Mid-Profe...",['English (USA)'],"English (USA),",30 minutes,Approximate Completion Time in minutes = 30,...,yes,yes,The.NET Framework 4.5 test measures knowledge ...,['Knowledge & Skills'],Name: .NET Framework 4.5\nDescription: The.NET...,30.0,False,Knowledge & Skills,"Professional Individual Contributor, Mid-Profe...",English (USA)
2,4094,.NET MVC (New),https://www.shl.com/products/product-catalog/v...,2026-05-08T10:39:53.276083+00:00,"['Mid-Professional', 'Professional Individual ...","Mid-Professional, Professional Individual Cont...",['English (USA)'],"English (USA),",17 minutes,Approximate Completion Time in minutes = 17,...,yes,no,Multi-choice test that measures the knowledge ...,['Knowledge & Skills'],Name: .NET MVC (New)\nDescription: Multi-choic...,17.0,False,Knowledge & Skills,"Mid-Professional, Professional Individual Cont...",English (USA)
3,4099,.NET MVVM (New),https://www.shl.com/products/product-catalog/v...,2026-05-08T10:41:53.047159+00:00,"['Mid-Professional', 'Professional Individual ...","Mid-Professional, Professional Individual Cont...",['English (USA)'],"English (USA),",5 minutes,Approximate Completion Time in minutes = 5,...,yes,no,Multi-choice test that measures the knowledge ...,['Knowledge & Skills'],Name: .NET MVVM (New)\nDescription: Multi-choi...,5.0,False,Knowledge & Skills,"Mid-Professional, Professional Individual Cont...",English (USA)
4,4018,.NET WCF (New),https://www.shl.com/products/product-catalog/v...,2026-05-08T10:41:58.935422+00:00,"['Mid-Professional', 'Professional Individual ...","Mid-Professional, Professional Individual Cont...",['English (USA)'],"English (USA),",11 minutes,Approximate Completion Time in minutes = 11,...,yes,no,Multi-choice test that measures the knowledge ...,['Knowledge & Skills'],Name: .NET WCF (New)\nDescription: Multi-choic...,11.0,False,Knowledge & Skills,"Mid-Professional, Professional Individual Cont...",English (USA)


In [5]:
embeddings = np.load(
    "../data/embeddings/catalog_embeddings.npy"
)
embeddings.shape

(377, 384)

In [6]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
bm25 = BM25Retriever(df)
faiss = FaissRetriever(
    df,
    embeddings
)
hybrid = HybridRetriever(
    bm25,
    faiss
)

In [8]:
query = "Java Spring SQL"
query_embedding = embedder.encode([query])

In [9]:
results = hybrid.search(
    query=query,
    query_embedding=query_embedding,
    top_k=10
)

In [10]:
for result in results:
    print("=" * 80)
    print(f"Name       : {result['name']}")
    print(f"Type       : {result['assessment_type']}")
    print(f"Job Level  : {result['job_level']}")
    print(f"Duration   : {result['duration']}")
    print(f"URL        : {result['url']}")
    print(f"Score      : {result['score']:.4f}")

Name       : Spring (New)
Type       : Knowledge & Skills
Job Level  : Mid-Professional, Professional Individual Contributor
Duration   : 9 minutes
URL        : https://www.shl.com/products/product-catalog/view/spring-new/
Score      : 9.3611
Name       : Java 8 (New)
Type       : Knowledge & Skills
Job Level  : Mid-Professional, Professional Individual Contributor
Duration   : 18 minutes
URL        : https://www.shl.com/products/product-catalog/view/java-8-new/
Score      : 6.8026
Name       : SQL Server (New)
Type       : Knowledge & Skills
Job Level  : Mid-Professional, Professional Individual Contributor
Duration   : 11 minutes
URL        : https://www.shl.com/products/product-catalog/view/sql-server-new/
Score      : 6.7016
Name       : Java Design Patterns (New)
Type       : Knowledge & Skills
Job Level  : Mid-Professional, Professional Individual Contributor
Duration   : 5 minutes
URL        : https://www.shl.com/products/product-catalog/view/java-design-patterns-new/
Score     

In [11]:
queries = [
    "Python",
    "Leadership",
    "Java Spring SQL",
    "Sales Personality",
    "Graduate Cognitive Test",
]

In [12]:
for query in queries:
    print("\n")
    print("=" * 100)
    print(f"QUERY : {query}")
    print("=" * 100)
    query_embedding = embedder.encode([query])
    results = hybrid.search(
        query=query,
        query_embedding=query_embedding,
        top_k=5
    )
    for result in results:
        print(
            f"{result['name']:<45}"
            f"{result['assessment_type']:<25}"
            f"{result['score']:.3f}"
        )



QUERY : Python
Python (New)                                 Knowledge & Skills       9.409
Linux Programming (General)                  Knowledge & Skills       0.248
C++ Programming (New)                        Knowledge & Skills       0.235
C Programming (New)                          Knowledge & Skills       0.233
R Programming (New)                          Knowledge & Skills       0.226


QUERY : Leadership
OPQ Team Types and Leadership Styles Report  Personality & Behavior   6.616
OPQ Team Types & Leadership Styles Profile   Personality & Behavior   6.599
OPQ Leadership Report                        Personality & Behavior   6.511
Enterprise Leadership Report 2.0             Personality & Behavior   6.268
Enterprise Leadership Report 1.0             Personality & Behavior   6.267


QUERY : Java Spring SQL
Spring (New)                                 Knowledge & Skills       9.361
Java 8 (New)                                 Knowledge & Skills       6.803
SQL Server (New)        

In [13]:
results[0]

{'index': 128,
 'score': 7.002227983193463,
 'name': 'Graduate Scenarios Narrative Report',
 'url': 'https://www.shl.com/products/product-catalog/view/graduate-scenarios-narrative-report/',
 'assessment_type': 'Biodata & Situational Judgment',
 'job_level': 'Mid-Professional, Professional Individual Contributor, Manager, Graduate',
 'duration': nan}